In [ ]:
import errno
import json
import os
import sys
import time
from pathlib import Path
import csv

import pandas as pd
import requests
from dotenv import load_dotenv
from requests.auth import HTTPBasicAuth

load_dotenv()

True

In [2]:
base_url = "https://api.planet.com/data/v1"
orders_url = 'https://api.planet.com/compute/ops/orders/v2'
stats_url = "{}/stats".format(base_url)
quick_url = "{}/quick-search".format(base_url)

In [ ]:
BASE_PATH = Path('/projectnb/planet/PLSP')
TEST_BASE_PATH = Path('/projectnb/modislc/users/fache/data/planet/')

# raw imagery in TEST_BASE_PATH / raw
# per site data in TEST_BASE_PATH / raw / site
# includes quick search results, order metadata, order results, data dir (per chunk)

# Helpers


In [4]:
def p(data):
    print(json.dumps(data, indent=2))

In [ ]:
def create_geojson_file(row):
    return BASE_PATH / 'geojson' / f'{row["site"]}.geojson'

def create_raw_path(row):
    return BASE_PATH / 'raw' / row["site"]

# Load Site Metadata


In [6]:
metadata_df = pd.DataFrame()
metadata_df['site'] = ['Walnut_Gulch_Kendall_Grasslands', 'Willard_Juniper_Savannah', 'Mountainair_Pinyon-Juniper_Woodland', 'Santa_Rita_Grassland', 'Santa_Rita_Mesquite', 'Sevilleta_shrubland', 'Walnut_Gulch_Lucky_Hills_Shrub', 'ARM_Southern_Great_Plains_site-_Lamont']
metadata_df['geojson_file'] = metadata_df.apply(create_geojson_file, axis=1)
metadata_df['raw_path'] = metadata_df.apply(create_raw_path, axis=1)
# metadata_df['geometry'] = metadata_df.apply(load_geometry_from_geojson, axis=1)
# metadata_gdf = gpd.GeoDataFrame(metadata_df, geometry='geometry')

In [7]:
metadata_df.head(10)

,site,geojson_file,raw_path
0,Walnut_Gulch_Kendall_Grasslands,/projectnb/planet/PLSP/geojson/Walnut_Gulch_Ke...,/projectnb/planet/PLSP/raw/Walnut_Gulch_Kendal...
1,Willard_Juniper_Savannah,/projectnb/planet/PLSP/geojson/Willard_Juniper...,/projectnb/planet/PLSP/raw/Willard_Juniper_Sav...
2,Mountainair_Pinyon-Juniper_Woodland,/projectnb/planet/PLSP/geojson/Mountainair_Pin...,/projectnb/planet/PLSP/raw/Mountainair_Pinyon-...
3,Santa_Rita_Grassland,/projectnb/planet/PLSP/geojson/Santa_Rita_Gras...,/projectnb/planet/PLSP/raw/Santa_Rita_Grassland
4,Santa_Rita_Mesquite,/projectnb/planet/PLSP/geojson/Santa_Rita_Mesq...,/projectnb/planet/PLSP/raw/Santa_Rita_Mesquite
5,Sevilleta_shrubland,/projectnb/planet/PLSP/geojson/Sevilleta_shrub...,/projectnb/planet/PLSP/raw/Sevilleta_shrubland
6,Walnut_Gulch_Lucky_Hills_Shrub,/projectnb/planet/PLSP/geojson/Walnut_Gulch_Lu...,/projectnb/planet/PLSP/raw/Walnut_Gulch_Lucky_...
7,ARM_Southern_Great_Plains_site-_Lamont,/projectnb/planet/PLSP/geojson/ARM_Southern_Gr...,/projectnb/planet/PLSP/raw/ARM_Southern_Great_...


# Load Planet Scene Counts


## Helpers


In [ ]:
def setup_filter(coords, minyear, maxyear):
    
    geometry_filter = {
        "type": "GeometryFilter",
        "field_name": "geometry",
        "config": {
          "type": "Polygon",
          "coordinates": coords
        }
    }
    
    date_filter = {
        "type": "DateRangeFilter",
        "field_name": "acquired", # date on which the "image was taken"
        "config":    {
            "gte": "{}-01-01T00:00:00.000Z".format(minyear),
            "lt":"{}-01-01T00:00:00Z".format(maxyear)
        }
    }
    
    ground_control =  {
        "type": "StringInFilter",
        "config": ["true"],
        "field_name": "ground_control" # NOTE
    }
    
    quality_category = {
        "type": "StringInFilter",
        "config": ["standard"],
        "field_name": "quality_category" # NOTE
    }
    
    cloud_cover =  {
        "type": "RangeFilter",
        "field_name": "cloud_cover",
        "config": {
            "gte": 0,
            "lte": 0.5
        } # NOTE
    }
    
    asset = {
        "type": "AssetFilter",
        "config": [
            "ortho_analytic_4b_sr", # NOTE before analytic_sr 
            "ortho_analytic_4b", # NOTE before analytic
            "ortho_udm2" # NOTE udm2 no longer exists, for instance, is only available globally through July 2018."
        ]
    }

    permission = {
        "type":"PermissionFilter",
        "config": [
            "assets:download" # NOTE
        ]
    }

    and_filter = {
        "type": "AndFilter",
        "config": [ 
            cloud_cover,
            quality_category,
            ground_control,
            date_filter,
            geometry_filter,
            permission,
            asset
        ]
    }
    
    return and_filter

In [ ]:
def read_geometry(path):
    
    if not os.path.exists(path):
        sys.exit("GeoJSON path doesn't exist: {}".format(path))#
        
    with open(path, "r") as f:
        geo = json.load(f)
        
    return geo

In [ ]:
def place_order(request, auth, order_name):
    headers = {'content-type': 'application/json'}
    
    response = requests.post(orders_url, data=json.dumps(request), auth=auth, headers=headers)
    print(response)
    print(response.reason)
    
    if response.status_code != 202:
        print("Order failed for {}".format(order_name))
        return -1
    
    order_id = response.json()['id']
    print(order_id)
    order_url = orders_url + '/' + order_id
    return order_url

## Go


In [ ]:
key = os.environ.get("PLANET_API_KEY")
print("key exists:", key is not None)
print("key length:", len(key) if key else 0)
# print("key repr:", repr(key))

PLANET_API_KEY = os.getenv('PLANET_API_KEY')

# Setup the session
session = requests.Session()
# Authenticate
session.auth = (PLANET_API_KEY, "")

print("GET:", session.get("https://api.planet.com/data/v1").status_code)

# Make a GET request to the Planet Data API
res = session.get(base_url)
# Response status code
if(res.status_code != 200):
    print("Cannot cannot to base server {} with status code {}".format(base_url, res.status_code))
    sys.exit("Cannot cannot to base server {} with status code {}".format(base_url, res.status_code))
else:
    print("Base server is alive.")

p(res.json())

key exists: True
key length: 36
key repr: 'PLAKa78d378db7e34593b495da57bca28f3b'
GET: 200
Base server is alive.
{
  "_links": {
    "_self": "https://api.planet.com/data/v1/",
    "asset-types": "https://api.planet.com/data/v1/asset-types/",
    "item-types": "https://api.planet.com/data/v1/item-types/",
    "spec": "https://api.planet.com/data/v1/spec"
  }
}


In [ ]:
output_dir = TEST_BASE_PATH / 'raw'

min_year = 2025
max_year = 2026

In [ ]:
for i, row in metadata_df.iterrows():
    print(f'\n{i=}')

    start_time = time.time()

    print(f'{"-"*10}{row["site"]}{"-"*10}')

    output_site_dir = os.path.join(output_dir, row['site'])
    if not os.path.exists(output_site_dir):
        try:
            os.makedirs(output_site_dir)
        except OSError as exc: # Guard against race condition
            if exc.errno != errno.EEXIST:
                raise
    
    print(f'{output_site_dir=}')

    geo = read_geometry(row['geojson_file'])
    
    for x in geo['features']: # get all geometry features, for each one # NOTE not really necessary since just one feature for each site
        feature_name = x['properties']['f']
        feature_coords = x['geometry']['coordinates']

        print(f'{feature_name=}')
        
        filter = setup_filter(feature_coords, min_year, max_year)
        
        # print("Filter config for search:")
        # p(filter)

        # ---------- get some quick stats
        
        print("---- feature count at year interval ----")
        request = {
            "interval" : "year",
            "filter" : filter,
            "item_types" : ["PSScene"] # NOTE replaces PSScene4Band https://community.planet.com/product-updates/event-psscene-migration-workshop-161
        }

        # Send the POST request to the API stats endpoint
        res = session.post(stats_url, json=request)

        # print(res.status_code)
        # print(res.text)
        # print(res.request.headers)
        # print(res.request.body)
        
        if res.status_code != 200:
            sys.exit("Stats search failed with code {}".format(res.status_code))

        for bucket in res.json()['buckets']:
            print("start_time: {} count: {}".format(bucket["start_time"], bucket["count"]))
        
        # ---------- perform real asset search

        print("---- quick search ----")
        request = {
            "filter" : filter,
            "item_types" : ["PSScene"]
        }

        # Send the POST request to the API quick search endpoint
        res = session.post(quick_url, json=request)
        
        if res.status_code != 200:
            sys.exit("Quick search failed with code {}".format(res.status_code))
            
        quick_search_results_json = res.json()
        
        filename = "{}_quick_search_result_{}_{}.json".format(feature_name.replace(" ", "_"), min_year, max_year)
        with open(os.path.join(output_site_dir, filename), 'w') as outfile:
            json.dump(quick_search_results_json, outfile)
            print('quick-search output file created: {}'.format(filename))
        
        # ---------- get all assets that need to be downloaded

        print('---- assembling feature ids to download ----')
        features = quick_search_results_json['features']

        if len(features) == 0:
            sys.exit("0 IDs returned in quick search.")

        id_list = []
        num_next_urls = 0
        while len(quick_search_results_json["features"]) > 0:
            print('iteration: {}'.format(num_next_urls))
            
            for x in quick_search_results_json["features"]: # go through all features and collect all scene ids
                id_list.append(x["id"])
            
            # Assign the "_links" -> "_next" property (link to next page of results) to a variable 
            next_url = quick_search_results_json["_links"]["_next"]
            if next_url is None:
                break
            num_next_urls += 1
            
            # from the next url, if there are results, update quick_search_results_json and append to output_site_dir
            time.sleep(5)
            res = session.get(next_url)
            
            if res.status_code != 200:
                sys.exit("Next page retrieval failed with code {}".format(res.status_code))
                
            quick_search_results_json = res.json()
            with open(os.path.join(output_site_dir, filename), 'a') as outfile:
                json.dump(quick_search_results_json, outfile)
            
            # output_site_dir is now on the next page, keep looping for more features
        
        # ---- end quick search loop

        print("total feature ids: {}".format(len(id_list)))
        print(f'{num_next_urls=}')

        print('---- chunking ids ----')
        chunks = [id_list[x:x+400] for x in range(0, len(id_list), 400)]
        
        print(f'{len(chunks)=}')
        
        if len(chunks) >= 80: # NOTE
            sys.exit("{} Chunks which is greater than 80.  This will exceed order capacity".format(len(chunks)))


        print("---- Checking connection with order server... ----")
        auth = HTTPBasicAuth(PLANET_API_KEY, '')
        response = requests.get(orders_url, auth=auth)
        
        if response.status_code != 200:
            sys.exit("Failed to connect to order server with code {}".format(response.status_code))
        else:
            print("connected!")
            
        orders_list = response.json()["orders"] # returns all previous orders created through my API key

        print('---- placing orders ----')
        # iterate through chunks (each is list of features (partial scenes))
        # create order per chunk
        # buffer coordinates
        # create order dir
        # place order

        orders_url_list = []
        bad_order_count = 0

        for chunk_num, chunk in enumerate(chunks):
            print(f'processing chunk {chunk_num}')

            order_name = "{}_chunk_{}_{}_{}".format(feature_name.replace(" ", "_"), chunk_num, min_year, max_year)

            # expand the coordinates by 0.015 degree rectangle
            # top left (lat, lon)
            # top right
            # bottom right
            # bottom left
            # top left
            feature_coords_buffer = feature_coords
            feature_coords_buffer[0][0][0] = feature_coords_buffer[0][0][0] - 0.0015
            feature_coords_buffer[0][0][1] = feature_coords_buffer[0][0][1] + 0.0015    
            feature_coords_buffer[0][1][0] = feature_coords_buffer[0][1][0] + 0.0015
            feature_coords_buffer[0][1][1] = feature_coords_buffer[0][1][1] + 0.0015
            feature_coords_buffer[0][2][0] = feature_coords_buffer[0][2][0] + 0.0015
            feature_coords_buffer[0][2][1] = feature_coords_buffer[0][2][1] - 0.0015
            feature_coords_buffer[0][3][0] = feature_coords_buffer[0][3][0] - 0.0015
            feature_coords_buffer[0][3][1] = feature_coords_buffer[0][3][1] - 0.0015
            feature_coords_buffer[0][4][0] = feature_coords_buffer[0][4][0] - 0.0015
            feature_coords_buffer[0][4][1] = feature_coords_buffer[0][4][1] + 0.0015

            request = {  
                "name": order_name,
                "order_type": "partial",
                "products": [
                    {  
                        "item_ids": chunk, # item ids belonging to this chunk, ids are the features found from the quick search
                        "item_type": "PSScene", # NOTE changed from PSScene4Band
                        "product_bundle": "analytic_sr_udm2, analytic_udm2" # NOTE changed from analytic_sr_udm2 analytic_sr
                        # analytic_udm2 contains [ortho_analytic_4b, ortho_analytic_4b_xml, ortho_udm2]
                        # analytic_sr_udm2 contains [ortho_analytic_4b_sr, ortho_analytic_4b_xml, ortho_udm2]
                    }
                ],
                "tools": [
                    {
                        "clip": {     
                            "aoi": {
                                "type": "Polygon",
                                "coordinates": feature_coords_buffer
                            }
                        }
                    }
                ]
            }

            filename = "order_{}.json".format(order_name)
            with open(os.path.join(output_site_dir, filename), 'w') as outfile:
                json.dump(request, outfile)
                print('orders output file created: {}'.format(filename))
            
            # time.sleep(5)
            
            # order_result = place_order(request, auth, order_name)
            
            # if order_result == -1:
            #     bad_order_count = bad_order_count +1
            #     continue
            # else:
            #     print("order url: {}".format(order_result))
            #     orders_url_list.append(order_result)
              
        # ---- end chunking loop

        print("\n{} out of {} orders not placed successfully.".format(bad_order_count, len(chunks)))      
        
        if len(orders_url_list) == 0:
            sys.exit("No orders placed successfully.")

        # save order urls to csv
        filename = "{}_orders_urls_{}_{}.csv".format(feature_name.replace(" ", "_"), min_year, max_year)
        with open(os.path.join(output_site_dir, filename), 'w', newline='') as outfile:
            writer = csv.writer(outfile)
            writer.writerows([[url] for url in orders_url_list])
            print('order urls file created: {}'.format(filename))

    print("---- %s seconds ----" % (time.time() - start_time))

    # if i == 1:
    #     break


i=0
----------Walnut_Gulch_Kendall_Grasslands----------
output_site_dir='/projectnb/modislc/users/fache/planet/raw/Walnut_Gulch_Kendall_Grasslands'
feature_name='Walnut Gulch Kendall Grasslands'
---- stat count ----
start_time: 2025-01-01T00:00:00.000000Z count: 842
---- quick search ----
quick-search output file created: Walnut_Gulch_Kendall_Grasslands_quick_search_result_2025_2026.json
---- assembling feature ids to download ----
iteration: 0
iteration: 1
iteration: 2
iteration: 3
Total IDs: 842
num_next_urls=3
chunking ids
len(chunks)=3
---- Checking connection with order server... ----
connected!
---- placing orders ----
processing chunk 0
orders output file created: order_Walnut_Gulch_Kendall_Grasslands_chunk_0_2025_2026.json
processing chunk 1
orders output file created: order_Walnut_Gulch_Kendall_Grasslands_chunk_1_2025_2026.json
processing chunk 2
orders output file created: order_Walnut_Gulch_Kendall_Grasslands_chunk_2_2025_2026.json
---- 18.306132793426514 seconds ----

i=1
